# Adaptive KV-Cache Compression Evaluation — baseline excluded

Same evaluation as `evaluate_adaptive_compression_(1).ipynb` — loads the three saved LightGBM payloads, recreates the held-out test set, evaluates all 16 latency–memory constraint pairs, and produces dataset-specific performance graphs plus selection and violation analyses — **except `"baseline"` is never a selectable candidate.** It is still loaded, predicted on, and used for `assert payload["configs"] == CONFIGS` compatibility with the saved models, but it is masked out of every eligibility check, so the adaptive loop must always pick one of the 9 real compression methods, even when only baseline would have met the constraints.

**Expected model folder:** `MyDrive/KV_Cache_Models/`


## 1. Mount Google Drive and import packages


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import io
import re
from itertools import product
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from sklearn.model_selection import train_test_split


Mounted at /content/drive


## 2. Configuration


In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RS = 42
TEST_FRAC = 0.15
PERCENTILES = [25, 50, 75, 90]

MODEL_DIR = Path("/content/drive/MyDrive/KV_Cache_Models")
OUTPUT_DIR = MODEL_DIR / "adaptive_evaluation_no_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIGS = [
    "baseline",
    "kvquant_2bit", "kvquant_3bit", "kvquant_4bit",
    "h2o_20", "h2o_40", "h2o_60",
    "rocketkv_8x", "rocketkv_16x", "rocketkv_32x",
]

FNAME = {
    "baseline": "kvquant_baseline_full_precision",
    "kvquant_2bit": "kvquant_2bit",
    "kvquant_3bit": "kvquant_3bit",
    "kvquant_4bit": "kvquant_4bit",
    "h2o_20": "h2o_budget_20pct",
    "h2o_40": "h2o_budget_40pct",
    "h2o_60": "h2o_budget_60pct",
    "rocketkv_8x": "rocketkv_ratio_8x",
    "rocketkv_16x": "rocketkv_ratio_16x",
    "rocketkv_32x": "rocketkv_ratio_32x",
}

DATASETS = ["gsm8k", "arc_challenge", "hellaswag", "squad"]
DATASET_TITLES = {
    "gsm8k": "GSM8K",
    "arc_challenge": "ARC Challenge",
    "hellaswag": "HellaSwag",
    "squad": "SQuAD1.1",
}

# Load CSVs from Google Drive (the authoritative source every implementation
# notebook actually writes to -- already mounted in cell 2 above); the repo's
# 2048_sample_results2/ folder is just a checked-in COPY of it, kept as a
# fallback in case Drive doesn't have a file yet; raw GitHub is the last
# resort.
import os

RESULTS2_DIRS = ["2048_sample_results2", "../2048_sample_results2", "../../2048_sample_results2"]
DRIVE_DIR  = "/content/drive/MyDrive/KVQuant_v3_Results"
LOCAL_DIRS = [DRIVE_DIR] + RESULTS2_DIRS + ["Data", "../Data", "../../Data"]
REPO_RAW   = "https://raw.githubusercontent.com/yoshikodes/KVCacheCompression/main/2048_sample_results2"

def _local_dir():
    for d in LOCAL_DIRS:
        if os.path.isdir(d) and any(f.endswith("_per_prompt.csv") for f in os.listdir(d)):
            return d
    return None

_LOCAL = _local_dir()
print("Data source:", _LOCAL if _LOCAL else REPO_RAW)


## 3. Load measured data


In [ ]:
def _read_csv_by_name(filename):
    """Read one CSV by exact filename from _LOCAL or REPO_RAW. Raises
    FileNotFoundError (local) or requests.HTTPError (remote, typically 404)
    if it doesn't exist there -- callers use that to fall back."""
    if _LOCAL:
        return pd.read_csv(os.path.join(_LOCAL, filename))
    response = requests.get(f"{REPO_RAW}/{filename}", timeout=60)
    response.raise_for_status()
    return pd.read_csv(io.StringIO(response.text))


def read_result(config, dataset):
    filename = f"{FNAME[config]}_{dataset}_per_prompt.csv"
    try:
        frame = _read_csv_by_name(filename)
    except (FileNotFoundError, requests.exceptions.HTTPError):
        # Some KVQuant runs saved HellaSwag (and RULER) as two checkpoint-
        # safety batches instead of one combined file. IMPORTANT: each
        # batch's raw index column is LOCAL to that batch (both restart at
        # 0!), so naively concatenating gives every idx value two rows --
        # silently turning every downstream merge in build_wide() into a
        # many-to-many join that MULTIPLIES the row count instead of adding
        # to it. Offset batch2's raw index by batch1's row count so every
        # value is globally unique before concatenating -- this matches how
        # the batches were actually sliced (items[:N] then items[N:2N]).
        parts = []
        offset = 0
        for suffix in ("_batch1", "_batch2"):
            part = _read_csv_by_name(f"{FNAME[config]}_{dataset}{suffix}_per_prompt.csv")
            index_col = "question_index" if "question_index" in part.columns else "item_index"
            part[index_col] = part[index_col] + offset
            offset = part[index_col].max() + 1
            parts.append(part)
        frame = pd.concat(parts, ignore_index=True)
    index_col = "question_index" if "question_index" in frame.columns else "item_index"
    frame = frame.rename(columns={index_col: "idx"})
    if "prompt" not in frame.columns and "full_prompt" in frame.columns:
        frame = frame.rename(columns={"full_prompt": "prompt"})
    return frame


def build_wide(target):
    """One row per prompt and one target column per compression method. The
    stored "prompt" column IS the exact text that was fed to the model
    (already fully prefilled -- fewshot prefix, answer choices, everything),
    so it's taken as-is from a single reference config; no reconstruction
    needed."""
    frames = []
    for dataset in DATASETS:
        reference = read_result("kvquant_2bit", dataset)[["idx", "prompt"]]

        base = pd.DataFrame({
            "dataset": dataset,
            "idx": reference["idx"].values,
            "model_input": list(reference["prompt"]),
        })
        for config in CONFIGS:
            values = read_result(config, dataset)[["idx", target]].rename(
                columns={target: config}
            )
            base = base.merge(values, on="idx", how="inner", validate="one_to_one")
        frames.append(base)

    wide = pd.concat(frames, ignore_index=True)
    assert not wide[CONFIGS].isna().any().any()
    return wide


## 4. Feature engineering and prediction helpers


In [ ]:
# ---------------------------------------------------------------------------
# Needed to unpickle the three .joblib artifacts: each now stores ONE
# PerTargetLGBMRegressor (latency/memory) or PerTargetSafeLGBMClassifier
# (correctness) -- a single fitted object wrapping N independently-tuned
# per-config sub-models, each with its own hyperparameters chosen by that
# config's own cross-validation, instead of every config sharing one setting
# (previously via MultiOutputRegressor/MultiOutputClassifier, which can only
# clone one shared config per target). These custom classes must be
# re-declared, identically, in every kernel that later joblib.load()s an
# artifact containing them -- this is that re-declaration, copied verbatim
# from the training notebooks (01/02's engine cell and
# 03_correct_prob_lightgbm_kfold.ipynb).
# ---------------------------------------------------------------------------
import lightgbm as lgb
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin


class PerTargetLGBMRegressor(BaseEstimator, RegressorMixin):
    """ONE fitted object wrapping N independently-tuned LGBMRegressor models
    -- one per config -- each using hyperparameters chosen by that config's
    own cross-validation instead of every config sharing a single
    hyperparameter setting."""
    def __init__(self, per_target_params):
        self.per_target_params = per_target_params   # list of dicts, len == len(CONFIGS)

    def fit(self, X, Y):
        self.estimators_ = [lgb.LGBMRegressor(**params).fit(X, Y[:, j])
                             for j, params in enumerate(self.per_target_params)]
        return self

    def predict(self, X):
        return np.column_stack([est.predict(X) for est in self.estimators_])


class SafeLGBMClassifier(BaseEstimator, ClassifierMixin):
    """Thin LGBMClassifier wrapper with a constant-probability fallback for
    degenerate (single-class) CV folds -- some folds have zero examples of one
    class for a given config, where a real classifier can't be fit. Every
    LightGBM hyperparameter is an explicit __init__ argument (not **kwargs),
    which sklearn.clone()/get_params() require to work correctly. Always
    returns a proper (n_samples, 2) predict_proba, whether or not a real
    classifier was fit, so it drops in wherever an ordinary classifier
    would."""
    def __init__(self, num_leaves=31, min_child_samples=20, n_estimators=100,
                 learning_rate=0.1, subsample=1.0, colsample_bytree=1.0,
                 random_state=None, verbose=-1, n_jobs=-1):
        self.num_leaves = num_leaves
        self.min_child_samples = min_child_samples
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.random_state = random_state
        self.verbose = verbose
        self.n_jobs = n_jobs

    def _lgbm_params(self):
        return dict(num_leaves=self.num_leaves, min_child_samples=self.min_child_samples,
                    n_estimators=self.n_estimators, learning_rate=self.learning_rate,
                    subsample=self.subsample, colsample_bytree=self.colsample_bytree,
                    random_state=self.random_state, verbose=self.verbose, n_jobs=self.n_jobs)

    def fit(self, X, y):
        y = np.asarray(y).astype(int)
        self.classes_ = np.array([0, 1])
        if len(np.unique(y)) < 2:
            self._const_proba = float(y.mean())          # degenerate fold fallback
            self._model = None
        else:
            self._const_proba = None
            self._model = lgb.LGBMClassifier(**self._lgbm_params()).fit(X, y)
        return self

    def predict_proba(self, X):
        n = len(X)
        if self._model is None:
            p1 = np.full(n, self._const_proba)
        else:
            p1 = self._model.predict_proba(X)[:, 1]
        return np.column_stack([1 - p1, p1])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


class PerTargetSafeLGBMClassifier(BaseEstimator, ClassifierMixin):
    """ONE fitted object wrapping N independently-tuned SafeLGBMClassifier
    models -- one per config -- each using hyperparameters chosen by that
    config's own cross-validation instead of every config sharing a single
    hyperparameter setting."""
    def __init__(self, per_target_params):
        self.per_target_params = per_target_params   # list of dicts, len == len(CONFIGS)

    def fit(self, X, Y):
        Y = np.asarray(Y).astype(int)
        self.estimators_ = [SafeLGBMClassifier(**params).fit(X, Y[:, j])
                             for j, params in enumerate(self.per_target_params)]
        return self

    def predict_proba(self, X):
        return [est.predict_proba(X) for est in self.estimators_]   # list of (n,2) arrays

    def predict(self, X):
        return np.column_stack([est.predict(X) for est in self.estimators_])


# ---------------------------------------------------------------------------
# Exact feature engineering used during training
# ---------------------------------------------------------------------------
def prompt_features(prompt):
    text = str(prompt)
    tokens = text.split()
    token_count = max(len(tokens), 1)
    char_count = max(len(text), 1)
    return {
        "n_char": len(text),
        "n_tok": len(tokens),
        "ttr": len(set(tokens)) / token_count,
        "digit_ratio": sum(c.isdigit() for c in text) / char_count,
        "punct_ratio": sum(not c.isalnum() and not c.isspace() for c in text) / char_count,
        "upper_ratio": sum(c.isupper() for c in text) / char_count,
        "avg_tok": char_count / token_count,
        "num_count": len(re.findall(r"\d+", text)),
        "has_q": int("?" in text),
        "n_newline": text.count("\n"),
    }


DS_DUMMY_COLS = [f"ds_{dataset}" for dataset in DATASETS]


def feature_matrix(frame, feature_cols):
    features = pd.DataFrame([prompt_features(p) for p in frame["model_input"]])
    dummies = pd.get_dummies(frame["dataset"], prefix="ds").reindex(
        columns=DS_DUMMY_COLS, fill_value=0
    )
    matrix = pd.concat(
        [features.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1
    )
    # Keep labels to avoid repeated sklearn feature-name warnings.
    return matrix.reindex(columns=feature_cols, fill_value=0)


def regression_predictions(payload, x):
    # payload["model"] is now ONE PerTargetLGBMRegressor; .predict(x) already
    # returns a stacked (n_samples, 10) array, one column per config -- each
    # column's sub-model has its own independently-tuned hyperparameters.
    raw = payload["model"].predict(x)
    if payload.get("target_space") == "log1p":
        raw = np.expm1(raw)
    return raw[0]


def correctness_predictions(payload, x):
    # payload["model"] is now ONE PerTargetSafeLGBMClassifier;
    # .predict_proba(x) returns a list of 10 per-config (n_samples, 2)
    # arrays -- the degenerate single-class-fold fallback is handled inside
    # SafeLGBMClassifier itself, so there is no "kind" tag to branch on here.
    probas = payload["model"].predict_proba(x)
    return np.asarray([float(p[0, 1]) for p in probas])


def smallest_combined_violation(pred_latency, pred_memory, latency_limit, memory_limit):
    """Minimize sum of relative excess; zero means a constraint is satisfied."""
    latency_excess = np.maximum(pred_latency / latency_limit - 1.0, 0.0)
    memory_excess = np.maximum(pred_memory / memory_limit - 1.0, 0.0)
    return latency_excess + memory_excess


## 5. Load models and reproduce the split


In [ ]:
# ---------------------------------------------------------------------------
# DIAGNOSTIC: confirm every CSV has a unique idx per row before merging.
# build_wide() does an inner merge on "idx" for each of the 10 configs; if
# idx is ever duplicated within one CSV (e.g. a batch1/batch2 concatenation
# gone wrong), the merge silently becomes many-to-many and the row count
# multiplies instead of staying at one row per prompt.
# ---------------------------------------------------------------------------
for dataset in DATASETS:
    print(f"\n=== {dataset} ===")
    for config in CONFIGS:
        d = read_result(config, dataset)
        print(
            config,
            "| rows:", len(d),
            "| unique idx:", d["idx"].nunique(),
            "| duplicated rows:", d["idx"].duplicated().sum()
        )


In [ ]:
# ---------------------------------------------------------------------------
# Load models and reconstruct the identical train/test split
# ---------------------------------------------------------------------------
latency_model = joblib.load(MODEL_DIR / "reg_total_latency_lightgbm.joblib")
memory_model = joblib.load(MODEL_DIR / "reg_peak_memory_lightgbm.joblib")
correct_model = joblib.load(MODEL_DIR / "clf_correct_lightgbm.joblib")

for payload in (latency_model, memory_model, correct_model):
    assert payload["configs"] == CONFIGS, "Unexpected compression-method ordering"

print("Loading measured per-prompt data...")
latency_wide = build_wide("total_latency_sec")
memory_wide = build_wide("peak_memory_mb")
correct_wide = build_wide("correct")

# Verify that all metrics contain exactly the same prompt keys.
key_cols = ["dataset", "idx"]
latency_keys = set(map(tuple, latency_wide[key_cols].to_numpy()))
assert latency_keys == set(map(tuple, memory_wide[key_cols].to_numpy()))
assert latency_keys == set(map(tuple, correct_wide[key_cols].to_numpy()))

# The training notebooks split each dataset separately using these exact values.
train_parts, test_parts = [], []
for dataset, group in latency_wide.groupby("dataset"):
    train_part, test_part = train_test_split(
        group, test_size=TEST_FRAC, random_state=RS, shuffle=True
    )
    train_parts.append(train_part)
    test_parts.append(test_part)

train_df = pd.concat(train_parts).sample(frac=1, random_state=RS).reset_index(drop=True)
test_df = pd.concat(test_parts).sample(frac=1, random_state=RS).reset_index(drop=True)

memory_lookup = memory_wide.set_index(key_cols)[CONFIGS]
correct_lookup = correct_wide.set_index(key_cols)[CONFIGS]
latency_lookup = latency_wide.set_index(key_cols)[CONFIGS]


Loading measured per-prompt data...


## 6. Calculate training-only percentile thresholds


In [ ]:
# ---------------------------------------------------------------------------
# Percentile thresholds use measured TRAINING data only, across all 10 methods.
# ---------------------------------------------------------------------------
latency_percentiles = dict(zip(
    PERCENTILES,
    np.percentile(train_df[CONFIGS].to_numpy(dtype=float).ravel(), PERCENTILES),
))

train_keys = pd.MultiIndex.from_frame(train_df[key_cols])
training_memory_values = memory_lookup.loc[train_keys].to_numpy(dtype=float).ravel()
memory_percentiles = dict(zip(
    PERCENTILES,
    np.percentile(training_memory_values, PERCENTILES),
))

percentile_table = pd.DataFrame({
    "percentile": PERCENTILES,
    "latency_threshold_sec": [latency_percentiles[p] for p in PERCENTILES],
    "memory_threshold_mb": [memory_percentiles[p] for p in PERCENTILES],
})
percentile_table.to_csv(OUTPUT_DIR / "training_percentiles.csv", index=False)
print("\nTraining-only thresholds:")
print(percentile_table.to_string(index=False))



Training-only thresholds:
 percentile  latency_threshold_sec  memory_threshold_gb
         25               3.865812             0.002287
         50               8.315173             0.006226
         75              17.327303             0.029915
         90              26.187829             0.082642


## 7. Run all 16 constraint combinations


In [ ]:
# ---------------------------------------------------------------------------
# Adaptive evaluation: 16 threshold pairs x every held-out prompt.
# Predictions deliberately occur inside the prompt loop, as requested.
# ---------------------------------------------------------------------------
records = []

for latency_pct, memory_pct in product(PERCENTILES, PERCENTILES):
    latency_limit = latency_percentiles[latency_pct]
    memory_limit = memory_percentiles[memory_pct]
    print(f"Evaluating L{latency_pct}/M{memory_pct}... "
          f"(latency <= {latency_limit:.3f}s, memory <= {memory_limit:.1f}MB)")

    for _, prompt_row in test_df.iterrows():
        one_prompt = prompt_row.to_frame().T
        x_latency = feature_matrix(one_prompt, latency_model["feature_cols"])
        x_memory = feature_matrix(one_prompt, memory_model["feature_cols"])
        x_correct = feature_matrix(one_prompt, correct_model["feature_cols"])

        pred_latency = regression_predictions(latency_model, x_latency)
        # The real per-prompt CSVs store peak memory natively in MB, and
        # reg_peak_memory_lightgbm.joblib is trained directly on that column
        # -- no unit conversion needed anywhere in this notebook.
        pred_memory = regression_predictions(memory_model, x_memory)
        pred_correct = correctness_predictions(correct_model, x_correct)

        # "baseline" is NOT a candidate in this notebook -- masked out of
        # every eligibility check below, so the loop always ends up picking
        # one of the 9 real compression methods, even on prompts where only
        # baseline would have met both constraints.
        BASELINE_INDEX = CONFIGS.index("baseline")
        eligible_indices = np.flatnonzero(
            (pred_latency <= latency_limit) & (pred_memory <= memory_limit)
        )
        eligible_indices = eligible_indices[eligible_indices != BASELINE_INDEX]
        eligible_methods = [CONFIGS[i] for i in eligible_indices]

        if eligible_methods:
            # Drop each family's MILDEST compression level (h2o_60,
            # kvquant_4bit, rocketkv_8x) from consideration -- the
            # lightest-touch setting per family is treated as not worth a
            # dedicated pick. Run the accuracy model on whatever's left; if
            # that empties the eligible set entirely, fall back to picking
            # among the full (still baseline-free) eligible set instead of
            # discarding a constraint-satisfying prompt.
            MILDEST_PER_FAMILY = {"h2o_60", "kvquant_4bit", "rocketkv_8x"}
            candidate_indices = [i for i in eligible_indices
                                  if CONFIGS[i] not in MILDEST_PER_FAMILY]
            if not candidate_indices:
                candidate_indices = list(eligible_indices)
            chosen_index = max(candidate_indices, key=lambda i: pred_correct[i])
            used_fallback = False
        else:
            # No feasible non-baseline method: minimize normalized combined
            # constraint excess, with baseline excluded from consideration
            # even here (np.inf so it can never win the argmin below).
            violation_scores = smallest_combined_violation(
                pred_latency, pred_memory, latency_limit, memory_limit
            ).copy()
            violation_scores[BASELINE_INDEX] = np.inf
            best_score = violation_scores.min()
            tied = np.flatnonzero(np.isclose(violation_scores, best_score))
            # Correctness probability breaks exact violation-score ties.
            chosen_index = max(tied, key=lambda i: pred_correct[i])
            used_fallback = True

        chosen_method = CONFIGS[chosen_index]
        key = (prompt_row["dataset"], prompt_row["idx"])
        actual_latency = float(latency_lookup.loc[key, chosen_method])
        actual_memory = float(memory_lookup.loc[key, chosen_method])
        actual_correct = int(correct_lookup.loc[key, chosen_method])

        records.append({
            "latency_percentile": latency_pct,
            "memory_percentile": memory_pct,
            "constraint_label": f"L{latency_pct}/M{memory_pct}",
            "latency_threshold_sec": latency_limit,
            "memory_threshold_mb": memory_limit,
            "dataset": prompt_row["dataset"],
            "idx": prompt_row["idx"],
            "selected_method": chosen_method,
            "eligible_count": len(eligible_methods),
            "used_fallback": used_fallback,
            "predicted_latency_sec": float(pred_latency[chosen_index]),
            "predicted_memory_mb": float(pred_memory[chosen_index]),
            "predicted_correct_probability": float(pred_correct[chosen_index]),
            "actual_latency_sec": actual_latency,
            "actual_memory_mb": actual_memory,
            "actual_correct": actual_correct,
            "latency_violation": actual_latency > latency_limit,
            "memory_violation": actual_memory > memory_limit,
            "any_violation": (actual_latency > latency_limit) or (actual_memory > memory_limit),
        })

adaptive_results = pd.DataFrame(records)
adaptive_results.to_csv(OUTPUT_DIR / "adaptive_prompt_results.csv", index=False)


Evaluating L25/M25...
Evaluating L25/M50...
Evaluating L25/M75...
Evaluating L25/M90...
Evaluating L50/M25...
Evaluating L50/M50...
Evaluating L50/M75...
Evaluating L50/M90...
Evaluating L75/M25...
Evaluating L75/M50...
Evaluating L75/M75...
Evaluating L75/M90...
Evaluating L90/M25...
Evaluating L90/M50...
Evaluating L90/M75...
Evaluating L90/M90...


## 8. Shared plotting constants and helpers


In [ ]:
# ---------------------------------------------------------------------------
# Shared plotting constants and helpers -- used by every graph below, so
# each style/color/ordering choice is made exactly once instead of being
# redefined per graph.
# ---------------------------------------------------------------------------
FIXED_ORDER = [
    "baseline",
    "kvquant_4bit", "kvquant_3bit", "kvquant_2bit",
    "h2o_60", "h2o_40", "h2o_20",
    "rocketkv_8x", "rocketkv_16x", "rocketkv_32x",
]
CONSTRAINT_ORDER = [f"L{lp}/M{mp}" for lp in PERCENTILES for mp in PERCENTILES]
STRATEGY_ORDER = FIXED_ORDER + CONSTRAINT_ORDER

METHOD_LABELS = {
    "baseline": "Baseline (FP16)",
    "kvquant_2bit": "KVQuant 2-bit", "kvquant_3bit": "KVQuant 3-bit", "kvquant_4bit": "KVQuant 4-bit",
    "h2o_20": "H2O 20%", "h2o_40": "H2O 40%", "h2o_60": "H2O 60%",
    "rocketkv_8x": "RocketKV 8x", "rocketkv_16x": "RocketKV 16x", "rocketkv_32x": "RocketKV 32x",
}
METHOD_COLORS = {
    "baseline": "#7F7F7F",
    "kvquant_2bit": "#9ECAE1", "kvquant_3bit": "#4292C6", "kvquant_4bit": "#08519C",
    "h2o_20": "#FCBBA1", "h2o_40": "#FB6A4A", "h2o_60": "#CB181D",
    "rocketkv_8x": "#238B45", "rocketkv_16x": "#74C476", "rocketkv_32x": "#C7E9C0",
}
FAMILY_REGIONS = [
    (-0.5, 0.5, "#FFF9DB", "Baseline"),
    (0.5, 3.5, "#E8F2FF", "KVQuant"),
    (3.5, 6.5, "#FFF0F0", "H2O"),
    (6.5, 9.5, "#EAF8EA", "RocketKV"),
]


def minmax(values):
    """Normalize values to [0, 1] (a constant array maps to all-0.5)."""
    values = np.asarray(values, dtype=float)
    lo, hi = values.min(), values.max()
    return np.full_like(values, 0.5) if np.isclose(lo, hi) else (values - lo) / (hi - lo)


def label_points(axis, x, normalized, actual, color, formatter, offset):
    """Annotate each point with its actual (non-normalized) value."""
    for xi, yi, value in zip(x, normalized, actual):
        axis.annotate(
            formatter(value), xy=(xi, yi), xytext=(0, offset), textcoords="offset points",
            ha="center", va="bottom" if offset >= 0 else "top", fontsize=7, color=color,
        )


## 9. Graph 1 — compression performance, per dataset and averaged


In [ ]:
# ---------------------------------------------------------------------------
# GRAPH 1 -- accuracy / memory / latency, normalized onto one axis, with
# background bands marking each compression family and the adaptive
# settings. One chart per dataset, plus one averaged equally across datasets.
# ---------------------------------------------------------------------------
def plot_combined_performance(data, title, save_path):
    data = data.copy()
    data["strategy"] = pd.Categorical(data["strategy"], categories=STRATEGY_ORDER, ordered=True)
    data = data.dropna(subset=["strategy"]).sort_values("strategy").reset_index(drop=True)
    data["strategy"] = data["strategy"].astype(str)

    x = np.arange(len(data))
    fig, ax = plt.subplots(figsize=(24, 9))

    regions = FAMILY_REGIONS + [(9.5, len(data) - 0.5, "#F3EEFF", "Adaptive algorithm")]
    for start, end, color, label in regions:
        ax.axvspan(start, end, color=color, alpha=0.75, zorder=0)
        ax.axvline(end, color="gray", linestyle="--", alpha=0.45)
        ax.text((start + end) / 2, 1.075, label, transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=13, fontweight="bold")

    series = [
        (data["accuracy"].to_numpy(), "#1f77b4", "o", "Actual accuracy", lambda v: f"{v:.3f}", 8),
        (data["average_memory_mb"].to_numpy(), "#2ca02c", "^", "Average actual memory", lambda v: f"{v:.3f} MB", 18),
        (data["average_latency_sec"].to_numpy(), "#d62728", "s", "Average actual latency", lambda v: f"{v:.2f}s", -18),
    ]
    for values, color, marker, name, fmt, offset in series:
        normalized = minmax(values)
        ax.plot(x, normalized, color=color, marker=marker, linewidth=2.5, markersize=7, label=name, zorder=3)
        label_points(ax, x, normalized, values, color, fmt, offset)

    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS.get(s, s) for s in data["strategy"]], rotation=65, ha="right", fontsize=9)
    ax.set_ylim(-0.15, 1.15)
    ax.set_yticks([0, 0.25, 0.50, 0.75, 1.0])
    ax.set_yticklabels(["low", "", "", "", "high"])
    ax.set_ylabel("Normalized metric value", fontsize=12)
    ax.set_xlabel("Compression method or adaptive constraint combination")
    ax.set_title(title, fontsize=17, fontweight="bold", pad=28)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False, fontsize=11)
    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


# One row per (dataset, strategy): fixed methods measured directly over the
# held-out test set; adaptive settings averaged over whichever method the
# loop actually chose for each prompt.
test_keys_by_dataset = {
    dataset: pd.MultiIndex.from_frame(test_df.loc[test_df.dataset == dataset, key_cols])
    for dataset in DATASETS
}
fixed_method_rows = [
    {
        "dataset": dataset, "strategy": method, "strategy_type": "Fixed method",
        "accuracy": float(correct_lookup.loc[test_keys_by_dataset[dataset], method].mean()),
        "average_memory_mb": float(memory_lookup.loc[test_keys_by_dataset[dataset], method].mean()),
        "average_latency_sec": float(latency_lookup.loc[test_keys_by_dataset[dataset], method].mean()),
    }
    for dataset in DATASETS for method in CONFIGS
]
adaptive_summary = (
    adaptive_results.groupby(["dataset", "constraint_label"], sort=False)
    .agg(accuracy=("actual_correct", "mean"), average_memory_mb=("actual_memory_mb", "mean"),
         average_latency_sec=("actual_latency_sec", "mean"))
    .reset_index().rename(columns={"constraint_label": "strategy"})
)
adaptive_summary["strategy_type"] = "Adaptive algorithm"

performance_summary = pd.concat([pd.DataFrame(fixed_method_rows), adaptive_summary], ignore_index=True)
performance_summary.to_csv(OUTPUT_DIR / "dataset_performance_summary.csv", index=False)

for dataset in DATASETS:
    plot_combined_performance(
        performance_summary[performance_summary["dataset"] == dataset],
        f"Compression Performance — {DATASET_TITLES[dataset]}",
        OUTPUT_DIR / f"combined_performance_{dataset}.png",
    )

# Average equally across datasets (every dataset counts the same, regardless
# of how many test prompts it has).
average_performance = (
    performance_summary.groupby(["strategy", "strategy_type"], as_index=False)
    .agg(accuracy=("accuracy", "mean"), average_memory_mb=("average_memory_mb", "mean"),
         average_latency_sec=("average_latency_sec", "mean"))
)
average_performance.to_csv(OUTPUT_DIR / "average_performance.csv", index=False)
plot_combined_performance(
    average_performance,
    "Compression Performance Averaged Equally Across All Datasets",
    OUTPUT_DIR / "average_performance_all_datasets.png",
)


## 10. Graph 2 — constraint-violation rates, per dataset and averaged


In [ ]:
# ---------------------------------------------------------------------------
# GRAPH 2 -- proportion of selected methods whose ACTUAL measured latency,
# memory, or either metric exceeded the constraint that chose them. One
# chart per dataset, plus one averaged equally across datasets.
# ---------------------------------------------------------------------------
def plot_violation_rates(data, title, save_path):
    long = data.melt(
        id_vars=["constraint_label"],
        value_vars=["latency_violation_rate", "memory_violation_rate", "any_violation_rate"],
        var_name="violation_type", value_name="proportion",
    )
    long["violation_type"] = long["violation_type"].map({
        "latency_violation_rate": "Latency violation",
        "memory_violation_rate": "Memory violation",
        "any_violation_rate": "Either or both",
    })
    long["constraint_label"] = pd.Categorical(long["constraint_label"], categories=CONSTRAINT_ORDER, ordered=True)
    long = long.sort_values("constraint_label")

    fig, axis = plt.subplots(figsize=(17, 7))
    sns.barplot(
        data=long, x="constraint_label", y="proportion", hue="violation_type",
        order=CONSTRAINT_ORDER, hue_order=["Latency violation", "Memory violation", "Either or both"],
        palette={"Latency violation": "#E45756", "Memory violation": "#4C78A8", "Either or both": "#B279A2"},
        ax=axis,
    )
    axis.set_ylim(0, 1)
    axis.set_xlabel("Latency percentile / Memory percentile")
    axis.set_ylabel("Proportion of test prompts")
    axis.set_title(title, fontsize=16, fontweight="bold")
    axis.tick_params(axis="x", rotation=45)
    axis.legend(title="Actual violation", loc="upper right")
    axis.grid(axis="y", alpha=0.25)
    for container in axis.containers:
        axis.bar_label(
            container, labels=[f"{bar.get_height():.1%}" if bar.get_height() > 0 else "" for bar in container],
            padding=2, fontsize=7, rotation=90,
        )
    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


dataset_violation_summary = (
    adaptive_results.groupby(["dataset", "latency_percentile", "memory_percentile", "constraint_label"], sort=False)
    .agg(latency_violation_rate=("latency_violation", "mean"), memory_violation_rate=("memory_violation", "mean"),
         any_violation_rate=("any_violation", "mean"), fallback_rate=("used_fallback", "mean"))
    .reset_index()
)
dataset_violation_summary.to_csv(OUTPUT_DIR / "dataset_constraint_violation_summary.csv", index=False)

for dataset in DATASETS:
    plot_violation_rates(
        dataset_violation_summary[dataset_violation_summary["dataset"] == dataset],
        f"Actual Constraint-Violation Rates — {DATASET_TITLES[dataset]}",
        OUTPUT_DIR / f"constraint_violations_{dataset}.png",
    )

# Average equally across datasets: each dataset already contributes exactly
# one row per constraint combo above, so averaging over "dataset" here gives
# every dataset the same weight regardless of its prompt count.
average_violations = (
    dataset_violation_summary.groupby(["latency_percentile", "memory_percentile", "constraint_label"], as_index=False)
    .agg(latency_violation_rate=("latency_violation_rate", "mean"), memory_violation_rate=("memory_violation_rate", "mean"),
         any_violation_rate=("any_violation_rate", "mean"), fallback_rate=("fallback_rate", "mean"))
)
average_violations.to_csv(OUTPUT_DIR / "average_constraint_violations.csv", index=False)
plot_violation_rates(
    average_violations,
    "Actual Constraint-Violation Rates Averaged Across All Datasets",
    OUTPUT_DIR / "average_constraint_violations_all_datasets.png",
)


## 11. Graph 3 — selection proportions, per dataset and averaged


In [ ]:
# ---------------------------------------------------------------------------
# GRAPH 3 -- for each of the 16 latency/memory constraint combinations, how
# often each compression method was actually selected. One 4x4 grid per
# dataset, plus one averaged equally across datasets.
# ---------------------------------------------------------------------------
def selection_proportions(results):
    """{(latency_pct, memory_pct): Series indexed by CONFIGS} of selection
    proportions within `results` (a slice of adaptive_results)."""
    combos = {}
    for lp in PERCENTILES:
        for mp in PERCENTILES:
            subset = results[(results["latency_percentile"] == lp) & (results["memory_percentile"] == mp)]
            combos[(lp, mp)] = subset["selected_method"].value_counts(normalize=True).reindex(CONFIGS, fill_value=0)
    return combos


def plot_selection_grid(combos, title, save_path, ylabel):
    fig, axes = plt.subplots(4, 4, figsize=(24, 23), sharex=False, sharey=True)
    for row, lp in enumerate(PERCENTILES):
        for col, mp in enumerate(PERCENTILES):
            axis = axes[row, col]
            proportions = combos[(lp, mp)]
            bars = axis.bar(range(len(CONFIGS)), proportions.values,
                             color=[METHOD_COLORS[m] for m in CONFIGS], edgecolor="black", linewidth=0.4)
            axis.set_title(f"Latency P{lp} / Memory P{mp}", fontsize=11, fontweight="bold")
            axis.set_ylim(0, 1)
            axis.set_xticks(range(len(CONFIGS)))
            axis.set_xticklabels([METHOD_LABELS[m] for m in CONFIGS], rotation=70, ha="right", fontsize=8)
            axis.tick_params(axis="x", which="both", labelbottom=True)
            axis.set_xlabel("Compression method", fontsize=9)
            if col == 0:
                axis.set_ylabel(ylabel, fontsize=9)
            axis.grid(axis="y", alpha=0.25)
            for bar, proportion in zip(bars, proportions.values):
                if proportion > 0:
                    axis.annotate(f"{proportion:.1%}", xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                                  xytext=(0, 3), textcoords="offset points", ha="center", va="bottom",
                                  fontsize=7, rotation=90)
    fig.suptitle(title, fontsize=18, fontweight="bold")
    fig.subplots_adjust(top=0.94, bottom=0.06, left=0.06, right=0.98, hspace=0.95, wspace=0.25)
    plt.savefig(save_path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {save_path}")


selection_rows = []
combos_by_dataset = {}
for dataset in DATASETS:
    combos = selection_proportions(adaptive_results[adaptive_results["dataset"] == dataset])
    combos_by_dataset[dataset] = combos
    plot_selection_grid(
        combos, f"Compression-Method Selection Proportions — {DATASET_TITLES[dataset]}",
        OUTPUT_DIR / f"selection_proportions_{dataset}.png", "Selection proportion",
    )
    for (lp, mp), proportions in combos.items():
        for method in CONFIGS:
            selection_rows.append({
                "dataset": dataset, "latency_percentile": lp, "memory_percentile": mp,
                "constraint_label": f"L{lp}/M{mp}", "compression_method": method,
                "selection_proportion": float(proportions[method]),
            })
pd.DataFrame(selection_rows).to_csv(OUTPUT_DIR / "dataset_selection_proportions.csv", index=False)

# Average equally across datasets, reusing the per-dataset proportions
# already computed above instead of recomputing them.
average_combos = {
    (lp, mp): sum(combos_by_dataset[d][(lp, mp)] for d in DATASETS) / len(DATASETS)
    for lp in PERCENTILES for mp in PERCENTILES
}
plot_selection_grid(
    average_combos, "Compression-Method Selection Proportions Averaged Across All Datasets",
    OUTPUT_DIR / "average_selection_proportions_all_datasets.png", "Average selection proportion",
)
average_selection_rows = [
    {"latency_percentile": lp, "memory_percentile": mp, "constraint_label": f"L{lp}/M{mp}",
     "selected_method": method, "selection_proportion": float(average_combos[(lp, mp)][method])}
    for lp in PERCENTILES for mp in PERCENTILES for method in CONFIGS
]
pd.DataFrame(average_selection_rows).to_csv(OUTPUT_DIR / "average_selection_proportions.csv", index=False)

print(f"\nAll graphs and CSV files were saved to: {OUTPUT_DIR}")
